# 1. Introducción

**Problema industrial:** Evaluación de confiabilidad operativa de línea de bombeo.

**Activo analizado:** PUMP101 y PUMP102 — línea de alimentación.

**Origen de datos:** Historial de eventos de falla registrados en PI / CMMS (simulado).

**Objetivo del análisis:** Calcular MTBF, MTTR y disponibilidad por equipo.


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"
from datetime import timedelta


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Cálculo de MTBF, MTTR y disponibilidad.

In [ ]:
eventos = pd.read_excel(EXCEL_DIR / "modelo_ingenieria.xlsx", sheet_name="Eventos", parse_dates=["Falla_Inicio", "Falla_Fin"])

metricas = []
for equipo, grp in eventos.groupby("Equipo"):
    grp = grp.sort_values("Falla_Inicio")
    mttr = grp["MTTR_horas"].mean()
    if len(grp) > 1:
        intervalos = grp["Falla_Inicio"].diff().dt.total_seconds() / 3600
        mtbf = intervalos.iloc[1:].mean()
    else:
        mtbf = np.nan
    disponibilidad = mtbf / (mtbf + mttr) if mtbf and not np.isnan(mtbf) else np.nan
    metricas.append({"Equipo": equipo, "MTBF_horas": mtbf, "MTTR_horas": mttr, "Disponibilidad": disponibilidad})

resultados_export = pd.DataFrame(metricas)
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

resultados_export.set_index("Equipo")[["MTBF_horas", "MTTR_horas"]].plot(
    kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"]
)
axes[0].set_title("MTBF y MTTR por equipo")
axes[0].set_ylabel("Horas")
axes[0].tick_params(axis="x", rotation=0)

for equipo, grp in eventos.groupby("Equipo"):
    for _, row in grp.iterrows():
        axes[1].barh(equipo, row["MTTR_horas"], left=row["Falla_Inicio"].toordinal(), height=0.3, color="coral")
axes[1].set_title("Timeline de eventos de falla")


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="MTBF_MTTR", index=False)
    eventos.to_excel(writer, sheet_name="Eventos", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

PUMP101 presenta MTTR elevado asociado a fallas de rodamiento. Priorizar kit de repuesto en almacén y estandarizar procedimiento de cambio para reducir tiempo de reparación.
